# (2) Data Generation — Open-Source Pipeline

This chapter walks through the **open-source port** of the data-collection process used in the RNN-based performance-map prediction paper. The original work relied on a closed-source stack (Mentor MotorSolve for FEA + MATLAB for control strategies and post-processing); this version replaces both with **[pyFEMM](https://www.femm.info/wiki/pyFEMM)** + Python, so the entire dataset can be regenerated on any machine with FEMM installed.

The pipeline lives under [`data_gen/`](data_gen/) in this directory and is organised as four scripts:

| Phase | Script | Role |
|---|---|---|
| 1 | [`run_phase1.py`](data_gen/run_phase1.py) | Sparse 9-point FEMM characterisation → λ-fit, T-fit |
| 2 | [`run_phase2.py`](data_gen/run_phase2.py) | MTPA/FW/MTPV trajectory on the fits (no FEMM) |
| 3 | [`run_phase3.py`](data_gen/run_phase3.py) | FEMM at each (N⁎, T⁎) point + Cu/Fe losses |
| 4 | [`run_phase4.py`](data_gen/run_phase4.py) | Interpolate scatter into the dense η(N, T) map |

Plus a multi-design driver [`run_dataset.py`](data_gen/run_dataset.py) that iterates Latin-Hypercube samples through Phases 1–4 and assembles the master ML dataset.

## 3.1  The four-stage methodology (Mohammadi 2017)

All Stage → Phase mappings below match the paper's `RNN:3_dataCollection` flowchart.

```
Stage 1: Characterisation   →  Phase 1
  For 9 sparse (i_d, i_q) points on a 3×3 grid, solve FEA and
  extract phase-A,B,C flux linkages → (λ_d, λ_q) via Park.
  Fit (λ_d, λ_q, T) as 2nd-degree bivariate polynomials.

Stage 2: Control strategies →  Phase 2
  For each speed N⁎, find optimal (i_d, i_q) maximising T_em
  subject to current and voltage limits. The single grid-
  search formulation collapses MTPA, FW and MTPV into one solve.

Stage 3: Operating-point FEA →  Phase 3
  At each (N⁎, T⁎) target inside the envelope, re-run FEMM at
  the optimal current vector; compute Cu loss (analytical) and
  Fe loss (lumped Steinmetz on stator iron).

Stage 4: Post-processing    →  Phase 4
  Interpolate the 48 op-point η scatter onto a dense (N, T)
  grid → Fig-0-style efficiency map. Save HDF5 for ML.
```

Two deliberate deviations from the original notebook in this repository:

1. **9-point sparse characterisation, not 21×21 dense**. The LaTeX text explicitly states "these 9 d-q flux linkage points are used to construct flux linkage maps using 2nd-degree polynomials." The original notebook had ballooned to a 441-point brute-force sweep; we restored the original choice.
2. **Direct T-polynomial fit, not d-q formula**. For our rectangular-magnet IPM, the standard formula $T_{em} = \tfrac{3}{2}p(\lambda_d i_q - \lambda_q i_d)$ underestimates torque by 3–25× (it captures only the fundamental; rectangular magnets produce strong harmonics). We fit `T(i_d, i_q)` directly from the FEMM Maxwell-stress measurements instead.

## 3.2  Phase 1 — sparse characterisation

Phase 1 builds the 12-slot / 4-pole IPM geometry, performs a 9-point (i_d, i_q) FEMM sweep, and fits the three response surfaces (λ_d, λ_q, T).

**Choice of pole count.** The notebook upstream chose 12 slots / 10 poles with an `A C B A C B …` winding pattern. That combination is *internally inconsistent*: the four phase-A coils end up at electrical angles 0°, 90°, 180°, 270° with polarities +,−,+,−, so the EMF contributions cancel identically for any rotor angle. Switching to 4 poles makes the same winding produce $4 \cos\theta_e$ — a proper distributed winding.

**Rotor offset.** With phase-A coils at slots 0 / 3 / 6 / 9, an open-circuit analytical model gives $\lambda_a(\varphi_{PM}) = -4 \sin(2 \varphi_{PM})$ (maximum positive when $\varphi_{PM} = 135^\circ$ mech). We therefore offset every magnet by `rotor_offset_deg = 135` so PM aligns with the d-axis at $\theta_e = 0$.

Each FEMM solve takes about two seconds; the full Phase-1 sweep finishes in under 30 seconds wall-clock.

In [ ]:
# Load the raw 9 FEMM samples and the fit coefficients written by run_phase1.py.
import pandas as pd
import numpy as np
from pathlib import Path

out = Path('data_gen/outputs')
samples = pd.read_csv(out / 'phase1_samples.csv')
fits = np.load(out / 'phase1_fit_coeffs.npz')

samples[['i_d_A', 'i_q_A', 'lambda_d_Wb', 'lambda_q_Wb', 'torque_Nm', 'solve_s']]

The 9-point grid covers `i_d in {-100, -50, 0}` A peak and `i_q in {0, 50, 100}` A peak. Open-circuit (`i_d = i_q = 0`) gives $\lambda_d \approx +0.010$ Wb — the PM flux linkage — with a clean sign, confirming that the rotor offset and Park sign conventions are correctly aligned.

The polynomial fit recovers each surface from these 9 samples (6 coefficients per surface, $\geq 99\%$ R² for λ_d and λ_q, $\approx 96\%$ for T). The fitted λ-surfaces look like this:

```{figure} data_gen/outputs/phase1_flux_surfaces.png
---
name: fig-phase1-flux-surfaces
width: 95%
---
**Phase 1 output.** Side-by-side 3D surfaces of $\lambda_d(I_d, I_q)$ (left) and $\lambda_q(I_d, I_q)$ (right) on the 9 FEMM samples (black dots) and the fitted polynomial surface. Reproduces the structure of `Fig 5.PNG` from the LaTeX chapter: λ_d tilts downward as $I_d$ goes negative (field-weakening), λ_q rises with $I_q$ (q-axis inductance). Axes are in A_RMS (= peak / √2), flux in mWb.
```

## 3.3  Phase 2 — control-strategy envelope

Phase 2 solves the operating-point optimisation

$$
\max_{i_d, i_q} \; T_{em}(i_d, i_q) \quad \text{s.t.} \quad
i_d^2 + i_q^2 \le I_{max}^2, \quad
\bigl(R_s i_d - \omega_e \lambda_q\bigr)^2 + \bigl(R_s i_q + \omega_e \lambda_d\bigr)^2 \le V_{max}^2
$$

for each speed $N$ from zero to $N_{max}$. The current-circle alone is the **MTPA** region; both bind → **field weakening**; the voltage ellipse alone is the **MTPV** region. We solve all three at once by gridding $(i_d, i_q)$ on a 401×401 grid and picking the feasible argmax — sub-second wall-clock since each evaluation is just a polynomial.

In [ ]:
envelope = pd.read_csv(out / 'phase2_trajectory.csv')
envelope[['N_rpm', 'i_d_A', 'i_q_A', 'T_em_Nm', 'U_mag_V', 'regime']].iloc[::8]

```{figure} data_gen/outputs/phase2_envelope.png
---
name: fig-phase2-envelope
width: 95%
---
**Phase 2 output.** Left: torque-speed envelope with regime classification (blue = MTPA, orange = FW, green = MTPV). Middle: optimal current vector trajectory — starts at the MTPA point $(-67, +74)$ A on the current circle, slides along the V-ellipse during FW, then spirals inward in MTPV. Right: phase-voltage magnitude versus speed — linear ramp up to the base speed $N_b \approx 3000$ RPM, then capped at $V_{max} = 200$ V. The textbook IPM-drive shape.
```

## 3.4  Phase 3 — operating-point FEMM + losses

Phase 3 picks 48 target operating points $(N^\star, T^\star)$ inside the envelope (8 speeds × 6 torque levels per speed), inverse-maps each through the polynomial fits to get an initial $(i_d^\star, i_q^\star)$, then **re-runs FEMM** at that current vector to get the *true* $T_{em}$, $\lambda_d$, $\lambda_q$ plus the stator B-field amplitude needed for iron loss.

Losses:

- **Copper**, analytical: $P_{Cu} = \tfrac{3}{2} R_s (i_d^2 + i_q^2)$, using amplitude-invariant Park currents.
- **Iron**, lumped Steinmetz: $P_{Fe} = (k_h f B_{pk}^{a} + k_e f^2 B_{pk}^2) \cdot V_{iron} \cdot \rho_{steel}$, with $k_h$, $k_e$, $a$ from M-19 silicon-steel literature, $B_{pk}$ sampled at 24 azimuths around the stator back-iron, and $f$ the electrical frequency.

Efficiency: $\eta = P_{mech} / (P_{mech} + P_{Cu} + P_{Fe})$.

In [ ]:
points = pd.read_csv(out / 'phase3_eta_points.csv')
points[['N_rpm', 'T_em_Nm', 'i_d_A', 'i_q_A', 'B_peak_iron_T', 'P_cu_W', 'P_fe_W', 'eta']].iloc[::8]

Some observable physics in this table:

- At **low speed, high torque** (top-left of the operating area) Cu loss dominates by an order of magnitude over Fe loss → η collapses to $\sim 0.3$ because mechanical power is tiny while resistive heating is large.
- At **mid speed, mid torque** Cu and Fe losses are comparable and η climbs into the $0.95$–$0.98$ range — the classic IPM sweet spot.
- $B_{peak}$ saturates around 2 T at peak torque (deep into M-19's knee) — a real motor's iron loss in this regime would be higher than our linear-Steinmetz model predicts.

```{figure} data_gen/outputs/phase3_eta_scatter.png
---
name: fig-phase3-eta-scatter
width: 80%
---
**Phase 3 output.** Scatter of the 48 FEMM-evaluated operating points coloured by efficiency. Pattern is unmistakably IPM-like: dark (low-η) cluster at low speed / high torque, yellow plateau (η ≥ 95 %) across the mid-band, sharp drop near the envelope at high speed.
```

## 3.5  Phase 4 — efficiency map

Phase 4 takes the Phase-3 scatter, interpolates $\eta(N, T)$ onto a dense $200 \times 120$ grid with `scipy.interpolate.griddata` (linear, falling back to nearest-neighbour at convex-hull edges), masks the region above the Phase-2 envelope, and plots Fig-0-style contours.

In [ ]:
grid = np.load(out / 'phase4_eta_grid.npz')
print('Dense grid shape:', grid['eta'].shape)
print('N axis: %d -> %d RPM' % (grid['N_rpm'].min(), grid['N_rpm'].max()))
print('T axis: %.1f -> %.1f Nm' % (grid['T_Nm'].min(), grid['T_Nm'].max()))
print('eta range inside envelope: %.3f .. %.3f' % (
    np.nanmin(np.where(grid['envelope_mask'], grid['eta'], np.nan)),
    np.nanmax(np.where(grid['envelope_mask'], grid['eta'], np.nan)),
))

```{figure} data_gen/outputs/phase4_efficiency_map.png
---
name: fig-phase4-efficiency-map
width: 100%
---
**Phase 4 output.** Final efficiency map $\eta(N, T)$, contoured at 1 % steps in the high-η region and 5 % steps below 90 %. The black curve is the peak-torque/peak-power envelope from Phase 2; the white dots are the actual FEMM samples. Compare to **`Fig 0.png`** from the LaTeX chapter (Toyota Prius efficiency map): identical structure (flat-top constant-T, knee at the base speed, hyperbolic decay) at smaller absolute magnitudes because our magnets are narrower (8 mm tangential, ~17 % pole coverage) and the stack is thinner (30 mm).
```

## 3.6  Multi-design dataset via LHS

For RNN training the paper uses a *population* of motor designs — each one a 6-D vector in the design space, each one producing a normalised efficiency map. We sample these designs with a **Latin Hypercube** over the geometry knobs (stator OD, bore, slot depth, magnet length, magnet width, stack length) with rejection sampling against geometric feasibility (magnet must fit inside rotor, slot must leave back-iron, pocket arc ≤ 70 % of the pole pitch).

The driver [`run_dataset.py`](data_gen/run_dataset.py) runs Phases 1–4 per LHS sample and aggregates everything into `outputs/dataset/master_dataset.npz` in paper-style normalised form: each design's $\eta$ map is resampled onto a *common* $(N/N_{max}, T/T_{max})$ axis so the RNN can learn the *shape* of the map separately from the per-design peak values.

```bash
# Smoke test (3 designs, ~3-5 min on a typical Windows machine):
python -m data_gen.run_dataset --n 3

# Full sweep (30 designs, ~30 min):
python -m data_gen.run_dataset --n 30
```

In [ ]:
# Peek at the LHS sample design parameters and feasibility.
from data_gen.design_space import lhs_designs, param_names, geometry_from_params, is_feasible

designs = lhs_designs(n_designs=5, seed=42)
for k, p in enumerate(designs, 1):
    g = geometry_from_params(p)
    ok, _ = is_feasible(g)
    print(f'design {k}: {dict(zip(param_names(), [round(v, 1) for v in p]))}  feasible={ok}')

### Master dataset on disk

After running the LHS sweep on the training copy of the dataset (`outputs/dataset_train/`), the master NPZ contains **2511 designs** that all 8 phases succeeded on (out of 2800 sampled — 10.3% rejection rate from FEMM/COM failures and infeasible geometries). Quick summary:


In [ ]:
import numpy as np
from pathlib import Path

master = np.load(Path('data_gen/outputs/dataset_train/master_dataset.npz'),
                 allow_pickle=False)
n = master['design_params'].shape[0]
print(f"designs:           {n}")
print(f"param names:       {list(master['param_names'])}")
print(f"N_max range (rpm): {master['N_max_rpm'].min():.0f} .. "
      f"{master['N_max_rpm'].max():.0f}")
print(f"T_max range (Nm):  {master['T_max_Nm'].min():.2f} .. "
      f"{master['T_max_Nm'].max():.2f}")
eta = master['eta_norm_grids']
print(f"eta grid shape:    {eta.shape}")
print(f"envelope coverage: {(~np.isnan(eta)).mean()*100:.1f}% of cells")
print(f"eta range inside:  {np.nanmin(eta):.3f} .. {np.nanmax(eta):.3f}")


### Master dataset on disk

After running the LHS sweep on the training copy of the dataset (`outputs/dataset_train/`), the master NPZ contains **2511 designs** that all 8 phases succeeded on (out of 2800 sampled — 10.3% rejection rate from FEMM/COM failures and infeasible geometries). Quick summary:


In [ ]:
import numpy as np
from pathlib import Path

master = np.load(Path('data_gen/outputs/dataset_train/master_dataset.npz'),
                 allow_pickle=False)
n = master['design_params'].shape[0]
print(f"designs:           {n}")
print(f"param names:       {list(master['param_names'])}")
print(f"N_max range (rpm): {master['N_max_rpm'].min():.0f} .. "
      f"{master['N_max_rpm'].max():.0f}")
print(f"T_max range (Nm):  {master['T_max_Nm'].min():.2f} .. "
      f"{master['T_max_Nm'].max():.2f}")
eta = master['eta_norm_grids']
print(f"eta grid shape:    {eta.shape}")
print(f"envelope coverage: {(~np.isnan(eta)).mean()*100:.1f}% of cells")
print(f"eta range inside:  {np.nanmin(eta):.3f} .. {np.nanmax(eta):.3f}")


### Master-dataset format

After `run_dataset.py` finishes, `outputs/dataset/master_dataset.npz` contains the arrays consumed by the ML chapter (Phase 5):

| Array | Shape | Meaning |
|---|---|---|
| `design_params` | `(n_designs, 6)` | Geometry vector per design |
| `param_names`   | `(6,)` | Parameter names |
| `N_max_rpm`     | `(n_designs,)` | Peak speed per design |
| `T_max_Nm`      | `(n_designs,)` | Peak torque per design |
| `N_norm_grid`   | `(80,)` | Normalised speed axis $\in [0,1]$ |
| `T_norm_grid`   | `(60,)` | Normalised torque axis $\in [0,1]$ |
| `eta_norm_grids`| `(n_designs, 60, 80)` | Efficiency maps on the common normalised axes |

The paper trains **two** networks: one predicts `eta_norm_grids` from `design_params`; the other predicts `(N_max_rpm, T_max_Nm)` from `design_params`. Splitting these prevents the dominant peak-value variation from drowning out the subtler shape information.

## 3.7  Code map

Every script under [`data_gen/`](data_gen/) is small, single-purpose, and has its own log file under `data_gen/logs/`. The pieces:

| File | Role |
|---|---|
| `config.py` | Geometry, materials, winding pattern, machine limits — single source of truth |
| `geometry_ipm.py` | Builds the 12-slot/4-pole IPM in FEMM (stator, rotor, magnets, flux barriers, coil labels, BC) |
| `dq_excitation.py` | Park / inverse-Park transforms (amplitude-invariant, conventions documented inline) |
| `stage1_characterize.py` | 9-point FEMM sweep |
| `flux_map_fit.py` | 2nd-degree bivariate polynomial fits |
| `control_strategies.py` | `TrajectorySolver`: grid-search MTPA/FW/MTPV + inverse mapping for Phase 3 |
| `stage3_operating_points.py` | Per-op-point FEMM re-run + B-field sampling for Fe loss |
| `losses.py` | Analytical Cu loss + Steinmetz iron loss |
| `stage4_efficiency_map.py` | Scatter → dense grid + contour plot |
| `design_space.py` | LHS sampler over design parameters with rejection on infeasible geometries |
| `dataset_runner.py` | Orchestrates Phases 1–4 in one FEMM session per design |

All run scripts (`run_phase{1..4}.py`, `run_dataset.py`) are CLI-only: no notebook dependency.

---

*See [`plan_May22.md`](../plan_May22.md) for the full implementation journey, including the four convention bugs the original notebook had that took most of Phase 1 to chase down (rotor leakage, pole/winding mismatch, rotor offset, Park q-axis sign).*